In [1]:
import os
import re
from pathlib import Path

import pandas as pd

from semevalpolar.finetuning.instruct.predict import generate_predictions_jsonl
from typing import List

from semevalpolar.utils import get_project_root

/home/atharva20240519/polar-semeval-2026


# Using dev phase data

In [2]:
data_path = Path(os.path.join(get_project_root(), "data", "test_phase", "subtask1", "dev", "eng.csv"))
dev_df = pd.read_csv(data_path)
dev_df = dev_df.sample(frac=1, random_state=42)

In [3]:
records = dev_df["text"]
gold = dev_df["polarization"]

In [4]:
predictions = generate_predictions_jsonl(records)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Running inference: 100%|██████████| 160/160 [09:14<00:00,  3.47s/sample]


In [5]:
import json

labels = []
text = []
with open('predictions/dev_data/predictions.jsonl', 'r') as f:
	for line in f:
		try:
			json_obj = json.loads(line)
			labels.append(json_obj['extracted_label'])
			text.append(json_obj['prediction'])
		except json.JSONDecodeError:
			pass

In [12]:
comparison = pd.DataFrame({
	"Predictions": labels,
	"Ground Truth": gold
})

wrong = comparison[comparison["Predictions"] != comparison["Ground Truth"]]

In [8]:
len(comparison[comparison["Predictions"] != comparison["Ground Truth"]])

39

In [14]:
wrong[wrong["Predictions"] == 0]

,Predictions,Ground Truth
108,0,1
94,0,1
142,0,1
128,0,1
98,0,1
97,0,1
131,0,1
115,0,1
123,0,1
127,0,1


# Using training dataset

In [ ]:
data_path = Path("data/splits/val.jsonl")
with data_path.open("r", encoding="utf-8") as f:
	records = f.readlines()

In [ ]:
type(records)

In [ ]:
import re
import codecs
from typing import List

INPUT_RE = re.compile(
    r'Input:\s*(.*?)\s*Reasoning:',
    re.DOTALL
)

def get_all_inputs(records: List[str]) -> List[str]:
    cleaned: List[str] = []

    for text in records:
        m = INPUT_RE.search(text)
        if not m:
            cleaned.append("")
            continue

        # decode escaped sequences like \\n, \\t
        decoded = codecs.decode(m.group(1), "unicode_escape")

        # normalize all whitespace
        normalized = " ".join(decoded.split())

        cleaned.append(normalized)

    return cleaned


In [14]:
dataset = get_all_inputs(records)

In [ ]:
predictions = generate_predictions_jsonl(dataset)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Running inference:   9%|▉         | 30/322 [00:05<01:01,  4.75sample/s]

In [16]:
import torch

In [17]:
print(torch.cuda.is_available())

True


In [1]:
import json

labels = []
text = []
with open('predictions/predictions.jsonl', 'r') as f:
    for line in f:
        try:
            json_obj = json.loads(line)
            labels.append(json_obj['extracted_label'])
            text.append(json_obj['prediction'])
        except json.JSONDecodeError:
            pass

In [3]:
import json
import re

def extract_final_answers(file_path):
    final_answers = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                json_obj = json.loads(line)
                match = re.search(r'Final Answer:\n(\d+)', json_obj['text'])
                if match:
                    final_answers.append(int(match.group(1)))
            except json.JSONDecodeError:
                pass
    return final_answers

final_answers = extract_final_answers('data/splits/val.jsonl')

In [ ]:
comparison = pd.DataFrame({
	"Predictions": labels,
	"Ground Truth": final_answers
})

In [ ]:
len(comparison[comparison["Predictions"] != comparison["Ground Truth"]])

# 1 Example

In [ ]:
import re
import json

def extract_labels_jsonl(input_path, output_path):
    pattern = re.compile(r'Final Answer \(output ONLY 0 or 1\):\s*(\d+)')

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            obj = json.loads(line)

            # Prefer extracted_label if it already exists
            if "extracted_label" in obj:
                label = int(obj["extracted_label"])
            else:
                # Fallback: extract from prediction text
                text = obj.get("prediction", "")
                match = pattern.search(text)
                if match is None:
                    continue  # or raise an error if required
                label = int(match.group(1))

            fout.write(json.dumps({"label": label}) + "\n")

In [ ]:
extract_labels_jsonl(os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "predictions.jsonl"),
                     os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "gold.jsonl"))

In [ ]:
def predict(text):
	prompt = f"""Input:
            {text}

            Reasoning:

            """